# Train a custom YOLO model on Kaggle (free GPU)

Same steps as the Colab notebook, set up for Kaggle. Works for any YOLOv11-format dataset zip (cap, gloves, shoes ...).

**One-time setup (about 10 minutes)**
1. Sign in at kaggle.com and **verify your phone number** (Settings > Phone verification). Without it, GPU and Internet stay locked.
2. **Upload your dataset:** Create > New Dataset > drag in the zip (e.g. `cap_v3.zip`), give it a name, **Create**. Kaggle may unpack the zip by itself - this notebook handles both cases.
3. **New Notebook:** Create > New Notebook. Then File > Import notebook and pick this file.
4. In the right-hand panel (**Notebook options / Settings**):
   - **Accelerator:** GPU T4 x2 (or GPU P100). If one fails with a CUDA error, switch to the other.
   - **Internet:** On (needed to install Ultralytics and fetch the base model).
   - **Persistence:** Files only (optional).
5. **Add Input:** (right panel > Input > Add Input) and add your dataset from step 2.
6. Edit the settings in **cell 3**, then run the cells **top to bottom**.

**When it finishes** the file is in the **Output** panel (right side > Output > `/kaggle/working`). Download the `.pt` and copy it into `D:\priya\Live CCTV analytics\models\`.

Kaggle gives about 30 GPU hours a week. A session stops after 9 hours, so keep each run well under that.


In [ ]:
# 1) GPU check
import shutil, subprocess
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU FOUND -> right panel > Notebook options > Accelerator > GPU T4 x2 (or P100), then run again")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)


In [ ]:
# 2) Install the same Ultralytics version used on your PC (so the model loads there). Needs Internet ON.
!pip -q install "ultralytics==8.4.176"
import ultralytics; print("ultralytics", ultralytics.__version__)


In [ ]:
# 3) SETTINGS  <-- edit this cell
ZIP_NAME    = "cap_v3.zip"     # the zip file name you uploaded as a Kaggle dataset (cap_v3.zip | gloves_v3.zip | shoes_v2.zip)
OUTPUT_NAME = "cap_v3.pt"      # file name of the trained model you will download

EPOCHS = 60             # 40-80 is typical on GPU. Training stops early if it stops improving (patience).
IMGSZ  = 960            # cap: 960 | gloves: 640 | shoes: 960
BATCH  = 16
FRACTION = 1.0          # 1.0 = use all training images

# cap:    ['cap', 'no_cap']          gloves: ['gloves', 'no_gloves']          shoes: ['shoes', 'no_shoes']
CLASS_NAMES = ['cap', 'no_cap']
DROP_LEAKED_VALID = True   # remove validation images that are augmented copies of training images (gives an honest score)


In [ ]:
# 4) Find the dataset you added as input
import os, glob, shutil
from pathlib import Path
print("inputs:", [p.name for p in Path("/kaggle/input").iterdir()])
ZIPS = glob.glob(f"/kaggle/input/**/{ZIP_NAME}", recursive=True)
print("zip found:" if ZIPS else "no zip found (Kaggle unpacked it) - will look for the unpacked folder:", ZIPS[:1])


In [ ]:
# 5) Unpack/copy, find the dataset, write data.yaml, remove leaked validation images
import zipfile, yaml, collections

DS = Path("/kaggle/working/dataset")
if DS.exists():
    shutil.rmtree(DS)
DS.mkdir(parents=True)
if ZIPS:
    with zipfile.ZipFile(ZIPS[0]) as z:
        bad = [n for n in z.namelist() if n.startswith("/") or ".." in Path(n).parts]
        assert not bad, f"unsafe paths in zip: {bad[:3]}"
        z.extractall(DS)
else:
    found = [p.parent.parent for p in Path("/kaggle/input").rglob("images") if p.parent.name == "train"]
    assert found, "No zip and no train/images folder found in /kaggle/input - did you add the dataset as Input?"
    print("copying", found[0], "...")
    shutil.copytree(found[0], DS / found[0].name)      # /kaggle/input is read-only, so work on a copy

# the dataset root is the folder that contains train/images
roots = [p.parent.parent for p in DS.rglob("images") if p.parent.name == "train"]
assert roots, "Could not find a train/images folder"
ROOT = roots[0]
VAL = "valid" if (ROOT / "valid" / "images").exists() else "val"
print("dataset root:", ROOT, "| validation folder:", VAL)

src = yaml.safe_load((ROOT / "data.yaml").read_text())
names = src["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
if CLASS_NAMES:
    assert len(CLASS_NAMES) == len(names), f"CLASS_NAMES has {len(CLASS_NAMES)} names but the dataset has {len(names)} classes"
    names = list(CLASS_NAMES)
print("classes:", dict(enumerate(names)))      # <- check this is right before training

def base(p):  # Roboflow adds ".rf.<hash>" to augmented copies of the same source photo
    return Path(p).name.split(".rf.")[0]

tr_bases = {base(p) for p in glob.glob(str(ROOT / "train" / "images" / "*"))}
if DROP_LEAKED_VALID:
    removed = 0
    for img in glob.glob(str(ROOT / VAL / "images" / "*")):
        if base(img) in tr_bases:
            os.remove(img)
            lab = ROOT / VAL / "labels" / (Path(img).stem + ".txt")
            if lab.exists():
                lab.unlink()
            removed += 1
    print("validation images removed because the same photo is in training:", removed)

def stats(split):
    imgs = len(glob.glob(str(ROOT / split / "images" / "*")))
    per = collections.Counter()
    for f in glob.glob(str(ROOT / split / "labels" / "*.txt")):
        for line in open(f):
            t = line.split()
            if t:
                per[names[int(t[0])]] += 1
    return imgs, dict(per)

for s in ("train", VAL):
    n, per = stats(s)
    print(f"{s:6s}: {n} images, labelled objects per class: {per}")

DATA_YAML = "/kaggle/working/data.yaml"
Path(DATA_YAML).write_text(yaml.safe_dump({
    "path": str(ROOT), "train": "train/images", "val": f"{VAL}/images",
    "names": {i: n for i, n in enumerate(names)},
}))


In [ ]:
# 6) Train (about 30-60 min on a GPU for ~2000-5000 images)
from ultralytics import YOLO
model = YOLO("yolo11n.pt")   # small model: fast enough for your CPU PC at run time
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, fraction=FRACTION,
    device=0, workers=2, patience=20, project="/kaggle/working/runs", name="train", exist_ok=True,
)
BEST = Path(results.save_dir) / "weights" / "best.pt"
print("best weights:", BEST)


In [ ]:
# 7) Score per class  (look at every class, not just the "all" row)
m = YOLO(str(BEST))
metrics = m.val(data=DATA_YAML, imgsz=IMGSZ, device=0, plots=False)
print()
print(f"{'class':15s} {'mAP50':>7s}")
for idx, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"{names[int(idx)]:15s} {ap:7.3f}")
print(f"{'ALL':15s} {metrics.box.map50:7.3f}   (precision {metrics.box.mp:.3f}, recall {metrics.box.mr:.3f})")


In [ ]:
# 8) Look at a few predictions on validation photos
import matplotlib.pyplot as plt, cv2, random
imgs = glob.glob(str(ROOT / VAL / "images" / "*"))
random.seed(1)
pick = random.sample(imgs, min(6, len(imgs)))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, p in zip(axes.ravel(), pick):
    r = m.predict(p, imgsz=IMGSZ, conf=0.35, device=0, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# 9) Save the model where you can download it, and clean up the big dataset copy
OUT = Path("/kaggle/working") / OUTPUT_NAME
shutil.copy(BEST, OUT)
shutil.rmtree("/kaggle/working/dataset", ignore_errors=True)      # keep the Output panel small
shutil.rmtree("/kaggle/working/runs", ignore_errors=True)
print("saved:", OUT, round(OUT.stat().st_size / 1e6, 1), "MB")
print("Download it: right panel > Output > /kaggle/working >", OUTPUT_NAME, "> ... > Download")
from IPython.display import FileLink, display
display(FileLink(OUTPUT_NAME))


## After the download (on your PC)

1. Copy the `.pt` file into `D:\priya\Live CCTV analytics\models\`.
2. Check its class names: `python tools\inspect_model.py models\cap_v3.pt`
3. Send me the table printed by cell 7. I will switch the config to the new model.

If the session stops before cell 9, the files are lost unless you use **Save Version > Save & Run All (Commit)**, which keeps `/kaggle/working` as the notebook's Output.
